# Programming Assignment: Automated Data Validation

Welcome to the **Automated Data Validation** assignment!

In production data pipelines, silent data quality issues are among the hardest bugs to catch — a column that should never be null suddenly has 5 % missing values, a numeric field quietly exceeds its valid range, or duplicate IDs slip through unnoticed. **Automated validation checks** solve this by encoding your data expectations as executable assertions that run every time data flows through your pipeline.

We will build these checks **from scratch using only pandas** — no external frameworks required.

### Why Automated Validation Matters

| Problem | Silent Effect | Automated Check |
|---------|---------------|-----------------|
| Nulls in a join key | Missing rows after merge | **Null check** |
| Age = –5 or 999 | Skewed statistics, wrong model | **Range check** |
| Duplicate IDs | Double-counted records | **Uniqueness check** |
| Malformed emails | Failed sends, revenue loss | **Pattern/regex check** |
| Multiple rule violations | Cascading downstream errors | **Rule engine** |

### What You Will Implement

* **`validate_no_nulls`** — detect null values column by column
* **`validate_range`** — flag numeric values outside an allowed interval
* **`validate_uniqueness`** — count duplicate values in a column
* **`validate_regex`** — check string values against a regex pattern
* **`create_validation_report`** — orchestrate all checks from a declarative rule list

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding the Data](#1---understanding-the-data)
- [2 - Null Validation](#2---null-validation)
    - **[Exercise 1 - `validate_no_nulls`](#exercise-1---validate_no_nulls)**
- [3 - Range Validation](#3---range-validation)
    - **[Exercise 2 - `validate_range`](#exercise-2---validate_range)**
- [4 - Uniqueness Validation](#4---uniqueness-validation)
    - **[Exercise 3 - `validate_uniqueness`](#exercise-3---validate_uniqueness)**
- [5 - Pattern Validation](#5---pattern-validation)
    - **[Exercise 4 - `validate_regex`](#exercise-4---validate_regex)**
- [6 - Validation Rule Engine](#6---validation-rule-engine)
    - **[Exercise 5 - `create_validation_report`](#exercise-5---create_validation_report)**
- [7 - Visualizing Results](#7---visualizing-results)

<a name='imports'></a>
## Imports

In [ ]:
import re
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Understanding the Data

We will work with a synthetic **survey dataset** that has been deliberately corrupted with a controlled error rate. The dataset contains the following columns:

| Column | Type | Valid Range / Format | Injected Errors |
|--------|------|---------------------|-----------------|
| `respondent_id` | string | unique, format `R-XXXX` | duplicate IDs |
| `age` | int | 0 – 120 | nulls, out-of-range values |
| `score` | float | 0 – 100 | nulls, out-of-range values |
| `email` | string | `*@*.*` pattern | malformed addresses |
| `country` | string | any | nulls |

With `error_rate=0.1`, roughly 10 % of rows in each column will contain an injected problem. Your validation functions must detect these issues reliably.

In [ ]:
df = helper_utils.generate_survey_dataset(n_samples=300, error_rate=0.1, random_state=42)
print(f"Shape: {df.shape}")
print(f"\nColumn info:")
df.info()
print(f"\nBasic statistics:")
df.describe().round(2)

<a name='2---null-validation'></a>
## 2 - Null Validation

<a name='exercise-1---validate_no_nulls'></a>
### **Exercise 1 - `validate_no_nulls`**

**Your Task:**
Implement a function that inspects each requested column and reports how many null values it contains.

**Requirements:**
- Accept a `pd.DataFrame` and an optional `columns` list.
- When `columns` is `None`, check **all** columns.
- Return a dict mapping each column name to a sub-dict with keys `'passed'` (`bool`) and `'null_count'` (`int`).
- `'passed'` is `True` only when `null_count == 0`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. If `columns` is `None`, set it to `df.columns.tolist()`.
2. Create an empty `result` dict.
3. Loop over each column name in `columns`.
4. Compute `null_count = int(df[col].isna().sum())`.
5. Store `result[col] = {'passed': null_count == 0, 'null_count': null_count}`.
6. Return `result`.

</details>

In [ ]:
# GRADED FUNCTION: validate_no_nulls
def validate_no_nulls(df, columns=None):
    """
    Check each column for null values.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to check (default: all).

    Returns:
        dict: {column_name: {'passed': bool, 'null_count': int}}
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
null_report = validate_no_nulls(df)
print("Null Validation Report:")
for col, info in null_report.items():
    status = "\u2713 PASSED" if info['passed'] else "\u2717 FAILED"
    print(f"  {col}: {status} \u2014 {info['null_count']} nulls")

In [ ]:
unittests.exercise_1(validate_no_nulls)

<a name='3---range-validation'></a>
## 3 - Range Validation

<a name='exercise-2---validate_range'></a>
### **Exercise 2 - `validate_range`**

**Your Task:**
Implement a function that checks whether all non-null numeric values in a column fall within a specified `[min_val, max_val]` interval.

**Requirements:**
- Drop nulls before checking (null handling is Exercise 1's job).
- `min_val=None` means no lower bound; `max_val=None` means no upper bound.
- Return a dict with keys `'passed'` (`bool`), `'violations'` (`int`), and `'violation_pct'` (`float`, rounded to 2 decimal places).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Isolate non-null values: `col_data = df[column].dropna()`.
2. Start with a boolean mask of all `True`: `mask = pd.Series(True, index=col_data.index)`.
3. If `min_val is not None`, `AND` the mask with `col_data >= min_val`.
4. If `max_val is not None`, `AND` the mask with `col_data <= max_val`.
5. Count violations: `violations = int((~mask).sum())`.
6. Compute `violation_pct = round(violations / len(col_data) * 100, 2)` (guard against division by zero).

</details>

In [ ]:
# GRADED FUNCTION: validate_range
def validate_range(df, column, min_val=None, max_val=None):
    """
    Validate that numeric values fall within a specified range.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to validate.
        min_val (float): Minimum allowed value (None = no lower bound).
        max_val (float): Maximum allowed value (None = no upper bound).

    Returns:
        dict: {'passed': bool, 'violations': int, 'violation_pct': float}
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
age_report = validate_range(df, 'age', min_val=0, max_val=120)
score_report = validate_range(df, 'score', min_val=0, max_val=100)
print(f"Age range validation:   {age_report}")
print(f"Score range validation: {score_report}")

In [ ]:
unittests.exercise_2(validate_range)

<a name='4---uniqueness-validation'></a>
## 4 - Uniqueness Validation

<a name='exercise-3---validate_uniqueness'></a>
### **Exercise 3 - `validate_uniqueness`**

**Your Task:**
Implement a function that counts how many values in a column appear more than once (i.e., are duplicates).

**Requirements:**
- Use `pd.Series.duplicated()` — it marks every occurrence after the first as a duplicate.
- Return a dict with keys `'passed'` (`bool`) and `'duplicate_count'` (`int`).
- `'passed'` is `True` only when `duplicate_count == 0`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Call `df[column].duplicated()` — this returns a boolean Series where `True` means the value is a duplicate.
2. Sum the boolean Series to count duplicates: `duplicate_count = int(df[column].duplicated().sum())`.
3. Return `{'passed': duplicate_count == 0, 'duplicate_count': duplicate_count}`.

</details>

In [ ]:
# GRADED FUNCTION: validate_uniqueness
def validate_uniqueness(df, column):
    """
    Check for duplicate values in a column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to check for uniqueness.

    Returns:
        dict: {'passed': bool, 'duplicate_count': int}
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
id_report = validate_uniqueness(df, 'respondent_id')
print(f"ID uniqueness: {id_report}")

In [ ]:
unittests.exercise_3(validate_uniqueness)

<a name='5---pattern-validation'></a>
## 5 - Pattern Validation

<a name='exercise-4---validate_regex'></a>
### **Exercise 4 - `validate_regex`**

**Your Task:**
Implement a function that validates string values in a column against a regular expression pattern.

**Requirements:**
- Drop nulls before checking.
- Cast remaining values to `str` before matching.
- Use `pd.Series.str.match(pattern)` — this checks for a match **at the start** of each string.
- Return a dict with keys `'passed'` (`bool`) and `'violations'` (`int`).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Isolate non-null string values: `col_data = df[column].dropna().astype(str)`.
2. Apply the pattern: `matches = col_data.str.match(pattern)`.
3. Count non-matching rows: `violations = int((~matches).sum())`.
4. Return `{'passed': violations == 0, 'violations': violations}`.

**Tip:** `str.match` anchors at the start. To validate the full string (e.g. email format `.+@.+\..+`), make sure your pattern accounts for the entire value or use `str.fullmatch` if needed.

</details>

In [ ]:
# GRADED FUNCTION: validate_regex
def validate_regex(df, column, pattern):
    """
    Validate that string values in a column match a regex pattern.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to validate.
        pattern (str): Regex pattern each value must match.

    Returns:
        dict: {'passed': bool, 'violations': int}
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
email_report = validate_regex(df, 'email', pattern=r'.+@.+\..+')
print(f"Email format validation: {email_report}")

In [ ]:
unittests.exercise_4(validate_regex)

<a name='6---validation-rule-engine'></a>
## 6 - Validation Rule Engine

<a name='exercise-5---create_validation_report'></a>
### **Exercise 5 - `create_validation_report`**

**Your Task:**
Implement a **rule engine** that accepts a list of validation rule configurations and runs all four check types you built above, returning a unified structured report.

**Requirements:**
- Each rule dict must have at minimum a `'type'` key and a `'column'` key.
- Supported rule types: `'no_nulls'`, `'range'`, `'uniqueness'`, `'regex'`.
- For `'range'` rules, also read optional `'min_val'` and `'max_val'` keys.
- For `'regex'` rules, also read the `'pattern'` key.
- Return a `list` of result dicts, one per rule, each containing: `'rule'`, `'column'`, `'passed'`, `'violations'`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step-by-step guidance:**

1. Initialise an empty `report = []` list.
2. Loop over each `rule` dict in `rules`.
3. Extract `rule_type = rule['type']` and `column = rule['column']`.
4. Create a base `entry` dict: `{'rule': rule_type, 'column': column, 'passed': True, 'violations': 0}`.
5. Dispatch to the correct validator based on `rule_type`:
   - `'no_nulls'` → call `validate_no_nulls(df, [column])`, read `res[column]`.
   - `'range'` → call `validate_range(df, column, min_val=rule.get('min_val'), max_val=rule.get('max_val'))`.
   - `'uniqueness'` → call `validate_uniqueness(df, column)`, read `res['duplicate_count']`.
   - `'regex'` → call `validate_regex(df, column, pattern=rule['pattern'])`.
6. Update `entry['passed']` and `entry['violations']` from the result.
7. Append `entry` to `report`.
8. Return `report`.

</details>

In [ ]:
# GRADED FUNCTION: create_validation_report
def create_validation_report(df, rules):
    """
    Run a list of validation rules and return a structured report.

    Args:
        df (pd.DataFrame): Input DataFrame.
        rules (list): List of rule dicts. Supported types:
            - {'type': 'no_nulls', 'column': str}
            - {'type': 'range', 'column': str, 'min_val': float, 'max_val': float}
            - {'type': 'uniqueness', 'column': str}
            - {'type': 'regex', 'column': str, 'pattern': str}

    Returns:
        list: List of result dicts with keys 'rule', 'column', 'passed', 'violations'.
    """
    ### START CODE HERE ###

    # YOUR CODE HERE
    raise NotImplementedError("Implement this function")

    ### END CODE HERE ###

In [ ]:
rules = [
    {'type': 'no_nulls', 'column': 'age'},
    {'type': 'range', 'column': 'age', 'min_val': 0, 'max_val': 120},
    {'type': 'range', 'column': 'score', 'min_val': 0, 'max_val': 100},
    {'type': 'uniqueness', 'column': 'respondent_id'},
    {'type': 'regex', 'column': 'email', 'pattern': r'.+@.+\..+'},
]

report = create_validation_report(df, rules)
report_df = pd.DataFrame(report)
print("Validation Report:")
print(report_df.to_string(index=False))

In [ ]:
unittests.exercise_5(create_validation_report)

<a name='7---visualizing-results'></a>
## 7 - Visualizing Results

In [ ]:
# Build validation_report dict format for visualization
validation_dict = {row['rule'] + '_' + row['column']: {'passed': row['passed'], 'violations': row['violations']}
                   for row in report}
helper_utils.visualize_validation_results(validation_dict)